## 03 — Prepare Real Data (Roboflow)

### 0. Setup

In [7]:
# Core imports
import os
import shutil
from pathlib import Path
from collections import Counter

import yaml

# ── Project root ──────────────────────────────────────────────────────────────
NOTEBOOK_DIR = Path(os.getcwd())
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
print(f"Project root: {PROJECT_ROOT}")

# ── Paths (all under data/) ────────────────────────────────────────────────────
DATA_DIR       = PROJECT_ROOT / 'data'
FINETUNE_DIR   = DATA_DIR / 'noodles_finetune_dataset'

# ── Shared constants ──────────────────────────────────────────────────────────
PIECE_LABELS = list('ABCDEFGHIJK')
NUM_CLASSES  = len(PIECE_LABELS)

PIECE_COLORS = {
    'A': ('Yellow',      (0xF9, 0xD6, 0x5E)),
    'B': ('SkyBlue',     (0x08, 0xA7, 0xE8)),
    'C': ('DarkBlue',    (0x20, 0x6D, 0xD9)),
    'D': ('Green',       (0x1F, 0xA1, 0x5B)),
    'E': ('Red',         (0xEE, 0x39, 0x4F)),
    'F': ('Teal',        (0x85, 0xDA, 0xBB)),
    'G': ('Pink',        (0xEC, 0x71, 0xA8)),
    'H': ('Purple',      (0xC7, 0x78, 0xB9)),
    'I': ('Orange',      (0xFC, 0x69, 0x0C)),
    'J': ('DarkRed',     (0xB6, 0x30, 0x48)),
    'K': ('YellowGreen', (0x95, 0xD4, 0x50)),
}

# Build unified class mapping
UNIFIED_NAMES = {
    i: f"{label}_{PIECE_COLORS[label][0]}"
    for i, label in enumerate(PIECE_LABELS)
}

print(f"\n✅ Setup complete")
print(f"   Fine-tune dir: {FINETUNE_DIR}")

Project root: /home/salumi/projects/project

✅ Setup complete
   Fine-tune dir: /home/salumi/projects/project/data/noodles_finetune_dataset


### 1. Download Real Data from Roboflow

In [8]:
# !pip install -q roboflow

from roboflow import Roboflow

# ⚠️  FILL IN your Roboflow project details below
ROBOFLOW_API_KEY   = "rtBCMfDQl6zSFnAQzsHn"
ROBOFLOW_WORKSPACE = "abdulsalams-workspace-cslqv"
ROBOFLOW_PROJECT   = "noodels"
ROBOFLOW_VERSION   = 2

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)

# Download in YOLOv8 segmentation format
dataset = project.version(ROBOFLOW_VERSION).download("yolov8")

REAL_DATA_DIR = Path(dataset.location)
print(f"\n✅ Real dataset downloaded to: {REAL_DATA_DIR}")
print(f"   Train: {REAL_DATA_DIR / 'train'}")
print(f"   Val:   {REAL_DATA_DIR / 'valid'}")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to noodels-2 in yolov8:: 100%|██████████| 214/214 [00:00<00:00, 8784.14it/s]



✅ Real dataset downloaded to: /home/salumi/projects/project/notebooks/noodels-2
   Train: /home/salumi/projects/project/notebooks/noodels-2/train
   Val:   /home/salumi/projects/project/notebooks/noodels-2/valid


### 2. Validate Roboflow Labels

In [9]:
# ── Read Roboflow's data.yaml ─────────────────────────────────────────────────
rf_yaml_path = REAL_DATA_DIR / 'data.yaml'
with open(rf_yaml_path) as f:
    rf_config = yaml.safe_load(f)

rf_names = rf_config.get('names', {})
if isinstance(rf_names, list):
    rf_names = {i: n for i, n in enumerate(rf_names)}

print(f"Unified target mapping ({NUM_CLASSES} classes):")
for k, v in UNIFIED_NAMES.items():
    print(f"  {k}: {v}")

print("\nRoboflow class mapping:")
for k, v in sorted(rf_names.items(), key=lambda x: int(x[0])):
    print(f"  {k}: {v}")

# ── Verify match ──────────────────────────────────────────────────────────────
remap = {}
mismatches = []

for rf_id, rf_name in rf_names.items():
    rf_id = int(rf_id)
    if rf_id in UNIFIED_NAMES:
        uni_name = UNIFIED_NAMES[rf_id]
        rf_letter = rf_name.strip().split('_')[0].upper()
        uni_letter = uni_name.split('_')[0].upper()
        if rf_letter == uni_letter:
            remap[rf_id] = rf_id
        else:
            mismatches.append(f"  ID {rf_id}: Roboflow='{rf_name}' vs Unified='{uni_name}'")
            remap[rf_id] = rf_id
    else:
        mismatches.append(f"  ID {rf_id}: '{rf_name}' — not in unified scheme (>10)")

if mismatches:
    print(f"\n⚠️  {len(mismatches)} potential mismatches:")
    for m in mismatches:
        print(m)
    print("\nProceeding anyway — class IDs will be kept as-is.")
else:
    print(f"\n✅ All {len(remap)} Roboflow classes match unified scheme perfectly!")
    print("   No remapping needed — labels will be copied as-is.")

Unified target mapping (11 classes):
  0: A_Yellow
  1: B_SkyBlue
  2: C_DarkBlue
  3: D_Green
  4: E_Red
  5: F_Teal
  6: G_Pink
  7: H_Purple
  8: I_Orange
  9: J_DarkRed
  10: K_YellowGreen

Roboflow class mapping:
  0: A_Yellow
  1: B_SkyBlue
  2: C_DarkBlue
  3: D_Green
  4: E_Red
  5: F_Teal
  6: G_Pink
  7: H_Purple
  8: I_Orange
  9: J_DarkRed
  10: K_YellowGreen

✅ All 11 Roboflow classes match unified scheme perfectly!
   No remapping needed — labels will be copied as-is.


### 3. Prepare Fine-Tune Dataset

In [10]:
FINETUNE_DIR.mkdir(parents=True, exist_ok=True)

stats = {'copied': 0, 'skipped_lines': 0, 'total_files': 0, 'total_images': 0}

for split_name, rf_split in [('train', 'train'), ('val', 'valid'), ('val', 'val')]:
    rf_img_dir = REAL_DATA_DIR / rf_split / 'images'
    rf_lbl_dir = REAL_DATA_DIR / rf_split / 'labels'

    if not rf_img_dir.exists() or not rf_lbl_dir.exists():
        continue

    out_img_dir = FINETUNE_DIR / 'images' / split_name
    out_lbl_dir = FINETUNE_DIR / 'labels' / split_name
    out_img_dir.mkdir(parents=True, exist_ok=True)
    out_lbl_dir.mkdir(parents=True, exist_ok=True)

    label_files = list(rf_lbl_dir.glob('*.txt'))
    print(f"\n{rf_split} → {split_name}: {len(label_files)} label files")

    for lbl_file in label_files:
        valid_lines = []
        with open(lbl_file) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 7:
                    stats['skipped_lines'] += 1
                    continue
                cls_id = int(parts[0])
                if 0 <= cls_id < NUM_CLASSES:
                    valid_lines.append(line.strip())
                    stats['copied'] += 1
                else:
                    stats['skipped_lines'] += 1

        out_lbl = out_lbl_dir / lbl_file.name
        with open(out_lbl, 'w') as f:
            f.write('\n'.join(valid_lines) + '\n' if valid_lines else '')
        stats['total_files'] += 1

        img_stem = lbl_file.stem
        for ext in ['.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG']:
            src_img = rf_img_dir / (img_stem + ext)
            if src_img.exists():
                dst_img = out_img_dir / src_img.name
                if not dst_img.exists():
                    shutil.copy2(src_img, dst_img)
                stats['total_images'] += 1
                break

print(f"\n{'='*50}")
print(f"✅ Dataset prepared!")
print(f"   Label files: {stats['total_files']}")
print(f"   Images:      {stats['total_images']}")
print(f"   Annotations: {stats['copied']}")
if stats['skipped_lines']:
    print(f"   ⚠️  Skipped:  {stats['skipped_lines']} invalid lines")


train → train: 104 label files

✅ Dataset prepared!
   Label files: 104
   Images:      104
   Annotations: 321


### 4. Create Fine-Tune Dataset YAML

In [11]:
finetune_yaml_content = f"""# IQ Noodles Fine-Tune Dataset — Real Photos (Phase 2)
# 11 classes (A-K), IDs 0-10 — matches Phase 1 synthetic training

path: {FINETUNE_DIR.resolve()}
train: images/train
val: images/val

nc: {NUM_CLASSES}

names:
"""
for idx, name in UNIFIED_NAMES.items():
    finetune_yaml_content += f"  {idx}: {name}\n"

finetune_yaml_path = FINETUNE_DIR / 'dataset.yaml'
with open(finetune_yaml_path, 'w') as f:
    f.write(finetune_yaml_content)

print(f"✅ Fine-tune dataset YAML: {finetune_yaml_path}")
print(finetune_yaml_content)

✅ Fine-tune dataset YAML: /home/salumi/projects/project/data/noodles_finetune_dataset/dataset.yaml
# IQ Noodles Fine-Tune Dataset — Real Photos (Phase 2)
# 11 classes (A-K), IDs 0-10 — matches Phase 1 synthetic training

path: /home/salumi/projects/project/data/noodles_finetune_dataset
train: images/train
val: images/val

nc: 11

names:
  0: A_Yellow
  1: B_SkyBlue
  2: C_DarkBlue
  3: D_Green
  4: E_Red
  5: F_Teal
  6: G_Pink
  7: H_Purple
  8: I_Orange
  9: J_DarkRed
  10: K_YellowGreen



### 5. Validate Labels

In [12]:
def validate_labels(dataset_dir, split='train'):
    """Check label files for correctness."""
    lbl_dir = Path(dataset_dir) / 'labels' / split
    img_dir = Path(dataset_dir) / 'images' / split

    label_files = sorted(lbl_dir.glob('*.txt'))
    image_files = sorted(img_dir.glob('*'))
    image_stems = {f.stem for f in image_files}

    print(f"  Images: {len(image_files)}  |  Labels: {len(label_files)}")

    class_counts = Counter()
    orphan_labels = []
    bad_lines = []
    total_annotations = 0

    for lf in label_files:
        if lf.stem not in image_stems:
            orphan_labels.append(lf.name)

        with open(lf) as f:
            for line_no, line in enumerate(f, 1):
                parts = line.strip().split()
                if not parts:
                    continue
                cls_id = int(parts[0])
                n_coords = len(parts) - 1

                if cls_id < 0 or cls_id >= len(UNIFIED_NAMES):
                    bad_lines.append(f"{lf.name}:{line_no} class_id={cls_id} out of range")
                if n_coords < 6 or n_coords % 2 != 0:
                    bad_lines.append(f"{lf.name}:{line_no} invalid polygon ({n_coords} coords)")
                else:
                    coords = [float(x) for x in parts[1:]]
                    if any(c < 0 or c > 1 for c in coords):
                        bad_lines.append(f"{lf.name}:{line_no} coords outside [0,1]")

                class_counts[cls_id] += 1
                total_annotations += 1

    print(f"  Total annotations: {total_annotations}")
    print(f"  Class distribution:")
    for cls_id in sorted(class_counts.keys()):
        name = UNIFIED_NAMES.get(cls_id, f"UNKNOWN_{cls_id}")
        print(f"    {cls_id:2d} ({name}): {class_counts[cls_id]}")

    if orphan_labels:
        print(f"\n  ⚠️  {len(orphan_labels)} label files without matching image")
    if bad_lines:
        print(f"\n  ❌ {len(bad_lines)} problematic lines:")
        for bl in bad_lines[:10]:
            print(f"    {bl}")
    else:
        print(f"\n  ✅ All labels valid!")

    return len(bad_lines) == 0

print("=== Train split ===")
train_ok = validate_labels(FINETUNE_DIR, 'train')
print("\n=== Val split ===")
val_ok = validate_labels(FINETUNE_DIR, 'val')

if train_ok and val_ok:
    print("\n✅ All labels validated — ready for fine-tuning!")
else:
    print("\n⚠️  Fix label issues above before proceeding")

=== Train split ===
  Images: 104  |  Labels: 104
  Total annotations: 321
  Class distribution:
     0 (A_Yellow): 24
     1 (B_SkyBlue): 27
     2 (C_DarkBlue): 25
     3 (D_Green): 35
     4 (E_Red): 36
     5 (F_Teal): 20
     6 (G_Pink): 29
     7 (H_Purple): 27
     8 (I_Orange): 25
     9 (J_DarkRed): 37
    10 (K_YellowGreen): 36

  ✅ All labels valid!

=== Val split ===
  Images: 0  |  Labels: 0
  Total annotations: 0
  Class distribution:

  ✅ All labels valid!

✅ All labels validated — ready for fine-tuning!
